# AcuDock Scout - Active Learning Virtual Screening

**Ranked #1 across all AcuDock approaches (30/40 total score)**

Traditional virtual screening docks **every** compound — for 1M compounds at
~30s each, that's ~347 days. AcuDock Scout uses **active learning** to find
the same top hits while docking **<10%** of the library.

## How It Works

```
Compound Library --> Bootstrap (random dock 2%) --> Train ML Surrogate
        --> Select via UCB (exploit + explore) --> Dock batch --> Retrain
        --> Converged? --> Report top hits + diversity analysis
```

Inspired by **HASTEN** (Graff et al., 2021): >90% of top hits found with <10% docking.

## Getting Started

1. Run the **install cell** (~2-3 min), runtime auto-restarts
2. Skip install, run remaining cells
3. Gradio interface launches — configure and start your campaign!

---

**License:** MIT | **Platform:** Google Colab | **Author:** AcuDock Project

In [ ]:
# === Step 1: Install Dependencies ===
!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel pdbfixer pandas numpy scipy scikit-learn matplotlib seaborn gradio

# Clone repo
!git clone https://github.com/Grimlock5310/AcuDock.git /content/AcuDock 2>/dev/null || (cd /content/AcuDock && git pull)

# Optional: Install Uni-Dock for GPU-accelerated docking (requires GPU runtime)
# Uncomment the next 2 lines for 1000x+ speedup on NVIDIA GPUs:
# !mkdir -p /content/unidock_env && wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/unidock_env 2>/dev/null && /content/unidock_env/bin/micromamba create -y -p /content/unidock_env/env -c conda-forge unidock && echo "Uni-Dock installed successfully" || echo "Uni-Dock install failed (GPU may not be available)"
# !ls /content/unidock_env/env/bin/unidock 2>/dev/null && echo "Uni-Dock binary found" || echo "Uni-Dock binary not found"

import os
os.kill(os.getpid(), 9)

## Launch AcuDock Scout

Run the cells below to start the active learning screening interface.

In [ ]:
# === Step 2: Imports ===
import warnings
warnings.filterwarnings('ignore')

import os, sys, io, time, random
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# Add Uni-Dock to PATH if installed via micromamba
if os.path.isdir('/content/unidock_env/env/bin'):
    os.environ['PATH'] = '/content/unidock_env/env/bin:' + os.environ['PATH']

sys.path.insert(0, '/content/AcuDock')
import acudock_utils as utils
from acudock_surrogate import SurrogateModel
from acudock_screening import BatchDockingManager, cluster_hits, compute_tanimoto_matrix

WORK_DIR = '/content/acudock_scout'
os.makedirs(WORK_DIR, exist_ok=True)

print('AcuDock Scout loaded.')
print(utils.get_docking_engine_status())

In [ ]:
# === Step 3: Launch Gradio Interface ===
import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')


def generate_demo_library():
    """Generate a demo compound library of ~500 SMILES."""
    seed_drugs = [
        ('Aspirin', 'CC(=O)Oc1ccccc1C(=O)O'),
        ('Ibuprofen', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
        ('Caffeine', 'Cn1c(=O)c2c(ncn2C)n(C)c1=O'),
        ('Acetaminophen', 'CC(=O)Nc1ccc(O)cc1'),
        ('Naproxen', 'COc1ccc2cc(ccc2c1)C(C)C(=O)O'),
        ('Metformin', 'CN(C)C(=N)NC(=N)N'),
        ('Omeprazole', 'COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C'),
        ('Atorvastatin', 'CC(C)c1n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c(c2ccc(F)cc2)c(c1c1ccccc1)C(=O)Nc1ccccc1'),
        ('Celecoxib', 'Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1'),
        ('Diclofenac', 'OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl'),
        ('Ciprofloxacin', 'O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O'),
        ('Loratadine', 'CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1'),
        ('Sildenafil', 'CCCc1nn(C)c2c1nc(nc2OCC)c1cc(ccc1OCC)S(=O)(=O)N1CCN(C)CC1'),
        ('Warfarin', 'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O'),
        ('Fluoxetine', 'CNCCC(Oc1ccc(C(F)(F)F)cc1)c1ccccc1'),
        ('Tamoxifen', 'CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1'),
        ('Metoprolol', 'COCCc1ccc(OCC(O)CNC(C)C)cc1'),
        ('Losartan', 'CCCCc1nc(Cl)c(n1Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1)CO'),
        ('Amlodipine', 'CCOC(=O)C1=C(COCCN)NC(C)=C(C1c1ccccc1Cl)C(=O)OC'),
        ('Captopril', 'CC(CS)C(=O)N1CCCC1C(=O)O'),
        ('Furosemide', 'NS(=O)(=O)c1cc(C(=O)O)c(NCc2ccco2)cc1Cl'),
        ('Clopidogrel', 'COC(=O)C(c1ccccc1Cl)N1CCc2sccc2C1'),
        ('Propranolol', 'CC(C)NCC(O)COc1cccc2ccccc12'),
        ('Lidocaine', 'CCN(CC)CC(=O)Nc1c(C)cccc1C'),
        ('Trimethoprim', 'COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC'),
        ('Chloroquine', 'CCN(CC)CCCC(C)Nc1ccnc2cc(Cl)ccc12'),
        ('Phenytoin', 'O=C1NC(=O)C(c2ccccc2)(c2ccccc2)N1'),
        ('Carbamazepine', 'NC(=O)N1c2ccccc2C=Cc2ccccc21'),
        ('Fluconazole', 'OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F'),
        ('Riluzole', 'Nc1nc2ccc(OC(F)(F)F)cc2s1'),
        ('Sorafenib', 'CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1'),
        ('Erlotinib', 'C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1'),
        ('Diphenhydramine', 'CN(C)CCOC(c1ccccc1)c1ccccc1'),
    ]

    library = list(seed_drugs)
    random.seed(42)

    # Generate enumeration variants
    for name, smi in seed_drugs:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        for _ in range(15):
            if len(library) >= 500:
                break
            try:
                order = list(range(mol.GetNumAtoms()))
                random.shuffle(order)
                renumbered = Chem.RenumberAtoms(mol, order)
                canon = Chem.MolToSmiles(Chem.MolFromSmiles(Chem.MolToSmiles(renumbered, canonical=False)))
                if canon and Chem.MolFromSmiles(canon) and canon not in [s for _, s in library]:
                    library.append((f'{name}_v{len(library)}', canon))
            except Exception:
                continue

    # Fragment-based additions
    scaffolds = ['c1ccccc1', 'c1ccncc1', 'c1ccc2ccccc2c1', 'C1CCCCC1',
                 'c1ccoc1', 'c1ccsc1', 'c1cnc2ccccc2n1', 'c1ccc2[nH]ccc2c1']
    subs = ['O', 'N', 'C(=O)O', 'F', 'Cl', 'OC', 'NC', 'C(F)(F)F',
            'C(=O)N', 'S(=O)(=O)N', 'CC(=O)O', 'OCC', 'C#N']
    for sc in scaffolds:
        for sub in subs:
            if len(library) >= 500:
                break
            test = f'{sc}{sub}'
            mol = Chem.MolFromSmiles(test)
            if mol:
                canon = Chem.MolToSmiles(mol)
                if canon not in [s for _, s in library]:
                    library.append((f'Frag_{len(library)}', canon))

    return library


def run_scout_campaign(pdb_id, residues_str, box_size, bootstrap_size,
                        batch_size, n_cycles, ucb_beta, exhaustiveness,
                        engine, library_text, progress=gr.Progress()):
    """Run full active learning campaign."""
    log_lines = []
    def log(msg):
        log_lines.append(msg)

    try:
        pdb_id = pdb_id.strip().upper()
        bootstrap_size = int(bootstrap_size)
        batch_size = int(batch_size)
        n_cycles = int(n_cycles)
        exhaustiveness = int(exhaustiveness)
        ucb_beta = float(ucb_beta)

        # Parse library
        if library_text.strip().lower() == 'demo' or not library_text.strip():
            log('Generating demo library (~500 compounds)...')
            compound_library = generate_demo_library()
        else:
            compound_library = []
            for line in library_text.strip().split('\n'):
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split(',', 1) if ',' in line else line.split('\t', 1)
                if len(parts) == 2:
                    name, smi = parts[0].strip(), parts[1].strip()
                else:
                    smi = parts[0].strip()
                    name = f'Cpd_{len(compound_library)+1}'
                if Chem.MolFromSmiles(smi):
                    compound_library.append((name, smi))

        if len(compound_library) < 20:
            return 'Error: Need at least 20 compounds.', None, None, None, None, None, None, None

        log(f'Library: {len(compound_library)} compounds')

        # Protein prep
        progress(0.02, desc='Preparing protein...')
        log(f'Preparing protein {pdb_id}...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        protein_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3
        log(f'  Center: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}]')

        # Init managers
        manager = BatchDockingManager(
            receptor_pdbqt=protein_pdbqt, center=center, box_size=box,
            exhaustiveness=exhaustiveness, n_poses=5, output_dir=WORK_DIR
        )
        library = manager.load_library(compound_library, shuffle=True)
        surrogate = SurrogateModel(fp_radius=2, fp_bits=2048, n_estimators=200)
        cycle_metrics = []

        # === Cycle 0: Bootstrap ===
        progress(0.05, desc='Bootstrap docking...')
        log(f'\n=== CYCLE 0: BOOTSTRAP ({bootstrap_size} compounds) ===')
        bootstrap = manager.random_sample(library, bootstrap_size)
        t0 = time.time()
        manager.dock_batch(bootstrap)
        bt = time.time() - t0
        stats = manager.get_statistics()
        log(f'  Docked: {stats["total_docked"]} in {bt:.0f}s')
        log(f'  Best: {stats["best_score"]} kcal/mol')

        # Train initial surrogate
        results_df = manager.get_all_results()
        metrics = surrogate.train(results_df['SMILES'].tolist(), results_df['Best_Score'].values)
        log(f'  Surrogate R2={metrics["train_r2"]}, RMSE={metrics["train_rmse"]}')
        cycle_metrics.append({
            'cycle': 0, 'n_docked': stats['total_docked'],
            'pct_library': stats['total_docked'] / len(library) * 100,
            'best_score': stats['best_score'],
            'train_r2': metrics['train_r2'], 'cv_r2': metrics['cv_r2_mean'],
            'train_rmse': metrics['train_rmse'],
            'n_strong': stats.get('n_strong_hits', 0),
            'n_moderate': stats.get('n_moderate_hits', 0),
        })

        # === AL Cycles ===
        for cycle in range(1, n_cycles + 1):
            pct_base = 0.05 + 0.75 * (cycle / n_cycles)
            progress(pct_base, desc=f'AL Cycle {cycle}/{n_cycles}...')
            log(f'\n=== CYCLE {cycle}/{n_cycles}: ACTIVE LEARNING ===')

            candidates = [smi for _, smi in library if smi not in manager.docked_smiles]
            if not candidates:
                log('All compounds docked. Stopping.')
                break

            selected = surrogate.select_next_batch(
                candidates, batch_size=min(batch_size, len(candidates)), beta=ucb_beta
            )
            smi_to_name = {smi: name for name, smi in library}
            batch_compounds = [(smi_to_name.get(smi, f'AL_{cycle}_{i}'), smi)
                               for i, (_, smi, _) in enumerate(selected)]

            log(f'  Selected {len(batch_compounds)} compounds via UCB')
            t0 = time.time()
            manager.dock_batch(batch_compounds)
            ct = time.time() - t0

            # Retrain
            results_df = manager.get_all_results()
            metrics = surrogate.train(results_df['SMILES'].tolist(), results_df['Best_Score'].values)
            stats = manager.get_statistics()
            pct_done = stats['total_docked'] / len(library) * 100

            log(f'  Docked: {stats["total_docked"]}/{len(library)} ({pct_done:.1f}%)')
            log(f'  Best: {stats["best_score"]} kcal/mol | R2={metrics["train_r2"]}')

            cycle_metrics.append({
                'cycle': cycle, 'n_docked': stats['total_docked'],
                'pct_library': pct_done, 'best_score': stats['best_score'],
                'train_r2': metrics['train_r2'], 'cv_r2': metrics['cv_r2_mean'],
                'train_rmse': metrics['train_rmse'],
                'n_strong': stats.get('n_strong_hits', 0),
                'n_moderate': stats.get('n_moderate_hits', 0),
            })

        # === Results ===
        progress(0.85, desc='Building results...')
        log(f'\n=== CAMPAIGN COMPLETE ===')
        final_stats = manager.get_statistics()
        log(f'Total docked: {final_stats["total_docked"]}/{len(library)}')
        log(f'Coverage: {final_stats["total_docked"]/len(library)*100:.1f}%')
        log(f'Best score: {final_stats["best_score"]} kcal/mol')
        log(f'Strong hits: {final_stats.get("n_strong_hits", 0)}')

        # Top hits table
        top_hits = manager.get_top_hits(n=20)
        display_df = top_hits[['Name', 'SMILES', 'Best_Score', 'MW', 'LogP']].copy()
        display_df.insert(0, 'Rank', range(1, len(display_df) + 1))
        display_df['SMILES'] = display_df['SMILES'].str[:40] + '...'

        # Convergence plot
        progress(0.9, desc='Generating plots...')
        metrics_df = pd.DataFrame(cycle_metrics)

        fig1, ax1 = plt.subplots(figsize=(8, 5))
        ax1.plot(metrics_df['pct_library'], metrics_df['best_score'],
                 'o-', color='#2196F3', linewidth=2, markersize=8, label='Best Score')
        ax1.set_xlabel('% Library Docked')
        ax1.set_ylabel('Best Score (kcal/mol)')
        ax1.set_title('Active Learning Convergence')
        ax1.invert_yaxis()
        ax1.axvline(x=10, color='red', linestyle=':', alpha=0.7, label='10% HASTEN target')
        ax1.legend()
        plt.tight_layout()
        conv_path = os.path.join(WORK_DIR, 'convergence.png')
        fig1.savefig(conv_path, dpi=120, bbox_inches='tight')
        plt.close(fig1)

        # Score distribution
        all_results = manager.get_all_results()
        bs_scores = all_results.head(bootstrap_size)['Best_Score'].dropna()
        al_scores = all_results.iloc[bootstrap_size:]['Best_Score'].dropna()

        fig2, ax2 = plt.subplots(figsize=(8, 5))
        if len(bs_scores) > 1:
            sns.kdeplot(data=bs_scores, ax=ax2, color='#90CAF9', fill=True,
                        alpha=0.4, label=f'Bootstrap (n={len(bs_scores)})')
        if len(al_scores) > 1:
            sns.kdeplot(data=al_scores, ax=ax2, color='#EF5350', fill=True,
                        alpha=0.4, label=f'AL-Selected (n={len(al_scores)})')
        ax2.axvline(x=-7, color='orange', linestyle='--', alpha=0.8, label='Moderate')
        ax2.axvline(x=-8, color='red', linestyle='--', alpha=0.8, label='Strong')
        ax2.set_xlabel('Vina Score (kcal/mol)')
        ax2.set_ylabel('Density')
        ax2.set_title('Score Distribution: Bootstrap vs AL')
        ax2.legend()
        plt.tight_layout()
        dist_path = os.path.join(WORK_DIR, 'distribution.png')
        fig2.savefig(dist_path, dpi=120, bbox_inches='tight')
        plt.close(fig2)

        # Surrogate performance
        fig3, ax3 = plt.subplots(figsize=(8, 5))
        ax3.plot(metrics_df['cycle'], metrics_df['train_r2'],
                 'o-', color='#2196F3', label='Train R2')
        ax3.plot(metrics_df['cycle'], metrics_df['cv_r2'],
                 's--', color='#FF9800', label='CV R2')
        ax3.set_xlabel('Cycle')
        ax3.set_ylabel('R2')
        ax3.set_title('Surrogate Model Performance')
        ax3.legend()
        ax3.set_ylim(-0.1, 1.1)
        plt.tight_layout()
        surr_path = os.path.join(WORK_DIR, 'surrogate.png')
        fig3.savefig(surr_path, dpi=120, bbox_inches='tight')
        plt.close(fig3)

        # 3D viewer of top hit
        progress(0.95, desc='Visualizing top hit...')
        best_hit = top_hits.iloc[0]
        viewer_html = ''
        try:
            best_smi = best_hit['SMILES']
            lig_pdbqt, _ = utils.prepare_ligand(best_smi, name='best_hit', output_dir=WORK_DIR)
            _, en, poses_path = utils.run_vina(
                protein_pdbqt, lig_pdbqt,
                center=center, box_size=box,
                exhaustiveness=32, n_poses=5
            )
            with open(protein_pdb, 'r') as f:
                prot_data = f.read()
            top_pose = utils.extract_pose_from_pdbqt(poses_path, 0)
            viewer_html = utils.make_3d_viewer_html(prot_data, top_pose)
            log(f'\nTop hit: {best_hit["Name"]} ({en[0][0]:.2f} kcal/mol refined)')
        except Exception as e:
            log(f'Visualization failed: {e}')

        # Save CSVs
        results_csv = os.path.join(WORK_DIR, 'all_results.csv')
        all_results.to_csv(results_csv, index=False)
        top_csv = os.path.join(WORK_DIR, 'top_hits.csv')
        top_hits.to_csv(top_csv, index=False)
        metrics_csv = os.path.join(WORK_DIR, 'cycle_metrics.csv')
        metrics_df.to_csv(metrics_csv, index=False)

        # Enrichment stats
        enrichment = ''
        if len(bs_scores) > 0 and len(al_scores) > 0:
            bs_rate = (bs_scores <= -7).mean() * 100
            al_rate = (al_scores <= -7).mean() * 100
            enrichment = (f'Bootstrap hit rate: {bs_rate:.1f}% | '
                         f'AL hit rate: {al_rate:.1f}% | '
                         f'Enrichment: {al_rate/max(bs_rate,0.01):.1f}x')

        progress(1.0, desc='Done!')
        return ('\n'.join(log_lines), display_df, metrics_df,
                conv_path, dist_path, surr_path, viewer_html, results_csv)

    except Exception as e:
        log(f'ERROR: {str(e)}')
        import traceback
        log(traceback.format_exc())
        return '\n'.join(log_lines), None, None, None, None, None, '', None


# ---------------------------------------------------------------------------
# Gradio Interface
# ---------------------------------------------------------------------------

with gr.Blocks(
    title='AcuDock Scout',
    theme=gr.themes.Soft(),
    css='.gradio-container { max-width: 1200px !important; }'
) as demo:

    gr.Markdown("""
    # AcuDock Scout
    **Active learning virtual screening — find top hits by docking <10% of your library.**
    """)

    with gr.Tabs():

        # === Campaign Tab ===
        with gr.Tab('Run Campaign'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### Target Protein')
                    s_pdb = gr.Textbox(label='PDB ID', value='1HSG')
                    s_residues = gr.Textbox(
                        label='Active Site Residues',
                        value='23,24,25,26,27,28,29,30'
                    )
                    s_box = gr.Slider(15, 40, value=20, step=5, label='Box Size (A)')

                    gr.Markdown('### Active Learning')
                    s_bootstrap = gr.Slider(50, 2000, value=100, step=50,
                                            label='Bootstrap Size')
                    s_batch = gr.Slider(25, 1000, value=50, step=25,
                                        label='Batch Size per Cycle')
                    s_cycles = gr.Slider(1, 20, value=3, step=1,
                                         label='AL Cycles')
                    s_beta = gr.Slider(0.5, 3.0, value=1.5, step=0.1,
                                       label='UCB Beta (explore vs exploit)')
                    s_exh = gr.Slider(4, 32, value=8, step=4,
                                      label='Exhaustiveness')
                    s_engine = gr.Radio(
                        ['Vina (CPU)', 'Uni-Dock (GPU)'],
                        value='Vina (CPU)', label='Docking Engine'
                    )

                    gr.Markdown('### Compound Library')
                    s_library = gr.Textbox(
                        label='Library (Name,SMILES per line or "demo")',
                        value='demo',
                        lines=5,
                        placeholder='Type "demo" for built-in ~500 compound library, or paste Name,SMILES lines'
                    )
                    s_btn = gr.Button('Start Campaign', variant='primary', size='lg')

                with gr.Column(scale=2):
                    s_log = gr.Textbox(label='Campaign Log', lines=15, interactive=False)

                    with gr.Tabs():
                        with gr.Tab('Top Hits'):
                            s_hits = gr.Dataframe(label='Top 20 Hits')
                        with gr.Tab('Cycle Metrics'):
                            s_metrics = gr.Dataframe(label='Metrics per Cycle')
                        with gr.Tab('Convergence'):
                            s_conv = gr.Image(label='Convergence Plot')
                        with gr.Tab('Score Distribution'):
                            s_dist = gr.Image(label='Bootstrap vs AL')
                        with gr.Tab('Surrogate Model'):
                            s_surr = gr.Image(label='R2 per Cycle')
                        with gr.Tab('Top Hit 3D'):
                            s_viewer = gr.HTML(label='Top Hit Pose')

                    s_csv = gr.File(label='Download All Results')

            s_btn.click(
                fn=run_scout_campaign,
                inputs=[s_pdb, s_residues, s_box, s_bootstrap, s_batch,
                        s_cycles, s_beta, s_exh, s_engine, s_library],
                outputs=[s_log, s_hits, s_metrics, s_conv, s_dist, s_surr, s_viewer, s_csv]
            )

        # === About Tab ===
        with gr.Tab('About'):
            gr.Markdown("""
            ### AcuDock Scout — Active Learning Virtual Screening

            **Algorithm:**
            1. **Bootstrap:** Dock random 2% of library to seed ML model
            2. **Train:** Random Forest on Morgan fingerprints + descriptors
            3. **Select:** UCB acquisition (exploit predicted good + explore uncertain)
            4. **Dock:** Dock selected batch with Vina or Uni-Dock
            5. **Repeat** until convergence or cycle limit

            **Key Parameters:**
            | Parameter | Description |
            |---|---|
            | Bootstrap Size | Initial random sample (2-5% of library) |
            | Batch Size | Compounds per AL cycle |
            | UCB Beta | Higher = more exploration, lower = more exploitation |
            | Exhaustiveness | Vina search thoroughness (8=fast, 32=thorough) |

            **References:**
            - HASTEN: Graff et al., *Chem. Sci.*, 2021
            - >90% of top hits found with <10% docking effort

            **Engines:**
            - **Vina (CPU):** Works on all runtimes
            - **Uni-Dock (GPU):** 1000x+ speedup for large libraries

            *MIT License | AcuDock Project*
            """)

demo.launch()